# Silver | Chamados

In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Normaliza texto para comparação: minúsculas, sem acento e sem espaços, pontos ou "_"
def normaliza(coluna):
    sem_acento = F.translate(F.lower(coluna), "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc")
    return F.regexp_replace(sem_acento, r"[^a-z0-9]", "")

# Load

As colunas de horário não são carregadas: vêm vazias na origem e os horários reais estão em `tb_chamados_hora`.

In [0]:
dbutils.widgets.text("bronze_table_name", "tb_chamados")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
df_chamados = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .select(
            "id_chamado",
            "id_cliente",
            "motivo",
            "canal",
            "resolvido",
            "tempo_espera",
            "tempo_atendimento",
            "id_atendente",
            "ingestion_datetime"
        ).distinct()
        .orderBy("id_chamado")
)

print(f"Linhas na bronze (sem repetições exatas): {df_chamados.count()}")
display(df_chamados.limit(5))

# Deduplicação

A bronze usa `append`: cada execução da ingestão adiciona uma nova cópia dos dados com o seu `ingestion_datetime`. Para cada `id_chamado` ficamos com a linha da ingestão mais recente. Como a data de ingestão só existe na bronze (decisão do grupo), ela é descartada aqui.

In [0]:
# Definição da janela agrupada pela chave primária e ordenada pelo timestamp decrescente
window_spec = Window.partitionBy("id_chamado").orderBy(F.col("ingestion_datetime").desc())

In [0]:
df_dedup = (
    df_chamados
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

# DF final deve ter 131838 linhas (chamados únicos na origem)
print(f"Total de linhas da tabela de chamados: {df_dedup.count()}")

# Tratamento

In [0]:
# Transforma ids em string
df_cast = cast_id_to_string(df_dedup)

In [0]:
# Valores de canal antes do tratamento
print(f"Grafias de canal na origem: {df_cast.select('canal').distinct().count()}")

In [0]:
# "canal": junta com o nome oficial de tb_canais, ignorando caixa, acento, pontuação e "_" ("Atend." vira "Atendimento")
df_ref_canais = (
    spark.table(f"{silver_schema}.tb_canais")
        .select(
            normaliza(F.col("nome_canal")).alias("chave"),
            F.col("nome_canal").alias("canal_oficial")
        )
)

chave_canal = normaliza(F.regexp_replace(F.col("canal"), r"(?i)^atend\.", "Atendimento"))

df_canal = (
    df_cast
    .withColumn("chave", chave_canal)
    .join(df_ref_canais, "chave", "left")
    .drop("chave")
)

In [0]:
dq_check(
    "tb_chamados",
    "canal deve existir em tb_canais",
    df_canal,
    F.col("canal_oficial").isNotNull()
)

In [0]:
# O que não casar com tb_canais fica em maiúsculas, como na tabela de referência
df_canal = (
    df_canal
    .withColumn("canal", F.coalesce(F.col("canal_oficial"), F.upper(F.trim(F.col("canal")))))
    .drop("canal_oficial")
)

# Deve restar apenas os 4 canais oficiais
display(df_canal.groupBy("canal").count().orderBy("canal"))

In [0]:
# "motivo": junta com o nome oficial de tb_motivos, ignorando caixa, acento e pontuação
df_ref_motivos = (
    spark.table(f"{silver_schema}.tb_motivos")
        .select(
            normaliza(F.col("nome_motivo")).alias("chave"),
            F.col("nome_motivo").alias("motivo_oficial")
        )
)

df_motivo = (
    df_canal
    .withColumn("chave", normaliza(F.col("motivo")))
    .join(df_ref_motivos, "chave", "left")
    .drop("chave")
)

In [0]:
dq_check(
    "tb_chamados",
    "motivo deve existir em tb_motivos",
    df_motivo,
    F.col("motivo_oficial").isNotNull()
)

In [0]:
df_motivo = (
    df_motivo
    .withColumn("motivo", F.coalesce(F.col("motivo_oficial"), F.trim(F.col("motivo"))))
    .drop("motivo_oficial")
)

# Deve restar apenas os 14 motivos oficiais
display(df_motivo.groupBy("motivo").count().orderBy(F.col("count").desc()))

In [0]:
# "resolvido": converte para booleano; o que não estiver na lista vira NULL
resolvido_norm = normaliza(F.col("resolvido"))

df_resolvido = df_motivo.withColumn(
    "resolvido",
    F.when(resolvido_norm.isin("sim", "s", "1"), True)
     .when(resolvido_norm.isin("nao", "n", "0", "naoresolvido"), False)
)

display(df_resolvido.groupBy("resolvido").count())

In [0]:
# "tempo_espera" e "tempo_atendimento": conversão para inteiro (o texto "NULL" da origem vira NULL)
df_tempos = (
    df_resolvido
    .withColumn("tempo_espera", F.expr("try_cast(tempo_espera as int)"))
    .withColumn("tempo_atendimento", F.expr("try_cast(tempo_atendimento as int)"))
)

In [0]:
# "id_atendente": ids de atendentes cadastrados em tb_atendentes
ids_atendentes = [r["id_atendente"] for r in spark.table(f"{silver_schema}.tb_atendentes").select("id_atendente").collect()]

In [0]:
# ids de atendentes inexistentes viram NULL
df_limpo = df_tempos.withColumn(
    "id_atendente",
    F.when(F.col("id_atendente").isin(ids_atendentes), F.col("id_atendente"))
)

In [0]:
df_chamados_hora = (
    spark.table(f"{bronze_schema}.tb_chamados_hora")
        .select("ID_Chamado", "Hora_Abertura_Chamado", "Hora_Inicio_Atendimento", "Hora_Finalizacao_Atendimento", "ingestion_datetime")
        .distinct()
)

window_spec_hora = Window.partitionBy("ID_Chamado").orderBy(F.col("ingestion_datetime").desc())
df_dedup_hora = df_chamados_hora.withColumn("rn", F.row_number().over(window_spec_hora)).filter(F.col("rn") == 1).drop("rn")

In [0]:
# Aplica as transformações e correções de datas
df_hora_tratado = cast_id_to_string(df_dedup_hora)
df_horas = (
    df_hora_tratado
    .withColumn("hora_abertura_chamado", parse_datetime(F.col("Hora_Abertura_Chamado")))
    .withColumn("hora_inicio_atendimento", parse_datetime(F.col("Hora_Inicio_Atendimento")))
    .withColumn("hora_finalizacao_atendimento", parse_datetime(F.col("Hora_Finalizacao_Atendimento")))
)

In [0]:
# Aplica correções de ano e virada de meia-noite
referencia = F.coalesce(F.col("hora_inicio_atendimento"), F.col("hora_finalizacao_atendimento"))
df_ano_corrigido = df_horas.withColumn(
    "hora_abertura_chamado",
    F.when(
        (F.year("hora_abertura_chamado") != F.year(referencia)) & (F.month("hora_abertura_chamado") == F.month(referencia)) & (F.dayofmonth("hora_abertura_chamado") == F.dayofmonth(referencia)),
        F.make_timestamp(F.year(referencia), F.month("hora_abertura_chamado"), F.dayofmonth("hora_abertura_chamado"), F.hour("hora_abertura_chamado"), F.minute("hora_abertura_chamado"), F.second("hora_abertura_chamado"))
    ).otherwise(F.col("hora_abertura_chamado"))
)

df_fim_corrigido = df_ano_corrigido.withColumn(
    "hora_finalizacao_atendimento",
    F.when(F.col("hora_finalizacao_atendimento") < F.col("hora_inicio_atendimento"), F.col("hora_finalizacao_atendimento") + F.expr("INTERVAL 1 DAY")).otherwise(F.col("hora_finalizacao_atendimento"))
)

df_tempos = df_fim_corrigido.withColumn("tempo_espera_segundos", F.unix_timestamp("hora_inicio_atendimento") - F.unix_timestamp("hora_abertura_chamado")) \
                            .withColumn("tempo_atendimento_segundos", F.unix_timestamp("hora_finalizacao_atendimento") - F.unix_timestamp("hora_inicio_atendimento")) \
                            .withColumnRenamed("ID_Chamado", "id_chamado")



In [0]:
# União das tabelas usando INNER JOIN
df_unificado = df_limpo.drop("tempo_espera", "tempo_atendimento").join(
    df_tempos.select("id_chamado", "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento", "tempo_espera_segundos", "tempo_atendimento_segundos"),
    on="id_chamado",
    how="inner"
)

In [0]:
dq_check("tb_chamados",
          "chaves após junção não podem ser nulas",
           df_unificado, F.col("id_chamado").isNotNull() & F.col("hora_abertura_chamado").isNotNull())

In [0]:
dq_check_unique("tb_chamados", "id_chamado único", df_unificado, ["id_chamado"])

In [0]:
dq_check(
    "tb_chamados",
    "id_atendente deve existir em tb_atendentes ou ser nulo",
    df_unificado,
    F.col("id_atendente").isNull() | F.col("id_atendente").isin(ids_atendentes)
)

In [0]:
dq_check(
    "tb_chamados",
    "tempos não podem ser negativos",
    df_unificado,
    (F.col("tempo_espera_segundos").isNull() | (F.col("tempo_espera_segundos") >= 0))
    & (F.col("tempo_atendimento_segundos").isNull() | (F.col("tempo_atendimento_segundos") >= 0))
)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_chamados")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
save_df(df_unificado, silver_table_name)